In [4]:
import numpy as np

Tarefa prática para os m étodos de solução de Sistema de Equações
Lineares.

Considere o seguinte sistema de equações lineares:$$\begin{cases}
16x_1 + 4x_2 + 8x_3 + 4x_4 = 32 \\
4x_1 + 10x_2 + 8x_3 + 4x_4 = 26 \\
8x_1 + 8x_2 + 12x_3 + 10x_4 = 38 \\
4x_1 + 4x_2 + 10x_3 + 12x_4 = 30
\end{cases}$$

a) Verifique se este sistema satisfaz as condições para utilização dos métodos de
Decomposção LU e Cholesky, explicando sua resposta.

In [5]:
#Função para verficar se o sistema satisfaz as condições da decomposição LU.
def verifca_LU(A,b):
    lengh = len(b)
    #Verficar se A é quadrada
    if A.shape[0] != A.shape[1]:
        print("A matriz A deve ser quadrada.")
        return False
    #verificar se os determinantes dos menores principais são diferentes de zero
    for i in range(1,lengh):
        #Python pega do índice 0 até o índice i-1 no slicing, pegando a matriz de ordem i
        if np.linalg.det(A[:i,:i]) == 0:
            print(f"O determinante do menor principal de ordem {i} é zero. A decomposição LU não é possível.")
            return False
    return True


In [6]:
#Função para verficar se o sistema satisfaz as condições da decomposição LU.
def verifica_Cholesky(A,b):
    lengh = len(b)
    #Verificar se A é simétrica
    A_transposta = A.T
    for i in range(lengh):
        for j in range(lengh):
            if A[i][j] != A_transposta[i][j]:
                print("A matriz A deve ser simétrica.")
                print(f"A[{i}][{j}] = {A[i][j]} != A^T[{i}][{j}] = {A_transposta[i][j]}")
                return False
    # if not np.allclose(A, A_transposta):
    #     print("A matriz A deve ser simétrica.")
    #     return False
    #Verificar se A é positiva definida
    # if not np.all(np.linalg.eigvals(A) > 0):
    #     print("A matriz A deve ser positiva definida.")
    #     return False
    for i in range(1,lengh):
    #Python pega do índice 0 até o índice i-1 no slicing, pegando a matriz de ordem i
        if np.linalg.det(A[:i,:i]) <= 0:
            print(f"O determinante do menor principal de ordem {i} é zero ou negativo. A decomposição LU não é possível.")
            return False
    return True

In [11]:
#Usando as funções:
Matriz_A = np.array([[16,4,8,4], [4,10,8,4], [8,8,12,10], [4,4,10,12]])
Matriz_b = np.array([[32,26,38,30]])

if verifca_LU(Matriz_A.copy(),Matriz_b.copy()):
  print("A decomposição LU é possível.")
else:
  print("A decomposição LU não é possível.")
if verifica_Cholesky(Matriz_A.copy(),Matriz_b.copy()):
  print("A decomposição de Cholesky é possível.")
else:
  print("A decomposição de Cholesky não é possível.")

A decomposição LU é possível.
A decomposição de Cholesky é possível.


Para a matriz A poder ser decomposta em LU ela deve ser uma matriz quadrada (mesmo número de linhas e colunas). Como também seu menores principais, constituídos das k primeiras linhas e k primerias colunas, devem ter determinante diferente de zero (det(Ak) != 0 para k=1,...,n-1).

Para a matriz A poder aplicada ao método de Cholesky ela deve ser simétrica  (aij = aji). Além disso, a matriz deve ser positiva definida, ou seja, os determinantes dos seus Ak menores pricipais devem ser maiores que zero (det(Ak) > 0 para k=1,...,n-1).

b) Utilizando os scripts desenvolvidos, obtenha solução numérica para esses sistema de equações lineares pelo método de Eliminação de Gauss e pelos métodos de Decomposição LU e Cholesky se forem viáveis.

Como visto no exercício a, os métodos de Decomposição Lu e Cholesky são viáveis para o sistema linear em questão.

In [12]:
#Método de Eliminação de Gauss
def eliminacao_gauss_aumentada(dim,A,b):
  #M é matriz aumentada [A|b]
  M = np.hstack((A, b.reshape(-1, 1)))

  for k in range(dim-1):
    for i in range(k+1, dim):
      m = M[i][k] / M[k][k]
      for j in range(k, dim + 1):  # +1 para incluir a coluna b
        M[i][j] = M[i][j] - m * M[k][j]

  # Substituição retroativa
  x = np.zeros(dim)
  for i in range(dim - 1, -1, -1):
    #A última coluna de M é o vetor b modificado
    x[i] = (M[i,dim] - np.sum(M[i,i+1:dim]*x[i+1:dim]))/M[i][i]
  return x

In [13]:
#Método da decomposição LU
def decomposicao_LU(A,b):
  L = np.zeros(A.shape)
  U = np.zeros(A.shape)
  lengh = len(b)
  for i in range(lengh):
    L[i,i] = 1.0

  for i in range(lengh):
    #Cálculo de U (tringular superior)
    for j in range(i,lengh):
      sum = 0.0
      for k in range(i):
        sum += L[i,k]*U[k,j]
      U[i,j] = A[i,j] - sum

    #Cálculo de L (tringular inferior)
    for j in range (i+1,lengh):
      sum_2 = 0.0
      for k in range(i):
        sum_2 += L[j,k]*U[k,i]
      L[j,i] = (A[j,i] - sum_2)/U[i,i]

  #Resolvendo Ly=b
  y = np.zeros(lengh)
  for i in range(lengh):
    sum_3 = 0.0
    for k in range(i):
      sum_3 += L[i,k]*y[k]
    y[i] = (b[i] - sum_3)/L[i][i]

  #Resolvendo Ux = y por substituição retroativa
  x = np.zeros(lengh)
  for i in range(lengh-1,-1,-1):
    sum_4 = 0.0
    for k in range(i+1,lengh):
      sum_4 += U[i,k]*x[k]
    x[i] = (y[i] - sum_4)/U[i,i]
  return x

In [14]:
#Método de Cholesky
def decomposicao_Cholesky(A,b):
  lengh = len(b)
  G = np.zeros((lengh,lengh))

  #Fazendo A = G*G^t
  for i in range (lengh):
    for j in range(i+1):
      soma=0.0
      if(i==j):
        for k in range(j):
          soma += G[i,k]**2
        G[i,j] = np.sqrt(A[i,i] - soma)
      else:
        for k in range(j):
          soma += G[i,k]*G[j,k]
        G[i,j]= (A[i,j] - soma)/G[j,j]

  #Resolvendo Gy=b
  y = np.zeros(lengh)

  for i in range (lengh):
    som_2 = 0.0
    for k in range (i):
      som_2 += G[i,k]*y[k]
    y[i] = (b[i] - som_2)/G[i][i]

  x = np.zeros(lengh)
  for i in range (lengh-1,-1,-1):
    som_3 = 0.0
    for k in range(i+1,lengh):
      som_3 += G[k,i]*x[k]
    x[i] = (y[i] - som_3)/G[i,i]
  return x


In [16]:
#Usando as funções:
Matriz_A = np.array([[16,4,8,4], [4,10,8,4], [8,8,12,10], [4,4,10,12]])
Matriz_b = np.array([32,26,38,30])

solução_gauss = eliminacao_gauss_aumentada(len(Matriz_b),Matriz_A.copy(),Matriz_b.copy())
solução_LU = decomposicao_LU(Matriz_A.copy(),Matriz_b.copy())
solução_Cholesky = decomposicao_Cholesky(Matriz_A.copy(),Matriz_b.copy())

print(f"Solução usando Eliminação de Gauss: {solução_gauss}")
print(f"Solução usando Decomposição LU: {solução_LU}")
print(f"Solução usando Decomposição Cholesky: {solução_Cholesky}")

Solução usando Eliminação de Gauss: [1. 1. 1. 1.]
Solução usando Decomposição LU: [1. 1. 1. 1.]
Solução usando Decomposição Cholesky: [1. 1. 1. 1.]


c) Se for possível resolver pelo método de Decomposição LU, apresente a matriz L e U obtida pelo m ́etodo.

In [17]:
def matrizes_L_U(A,b):
  L = np.zeros(A.shape)
  U = np.zeros(A.shape)
  lengh = len(b)
  for i in range(lengh):
    L[i,i] = 1.0

  for i in range(lengh):
    #Cálculo de U (tringular superior)
    for j in range(i,lengh):
      sum = 0.0
      for k in range(i):
        sum += L[i,k]*U[k,j]
      U[i,j] = A[i,j] - sum

    #Cálculo de L (tringular inferior)
    for j in range (i+1,lengh):
      sum_2 = 0.0
      for k in range(i):
        sum_2 += L[j,k]*U[k,i]
      L[j,i] = (A[j,i] - sum_2)/U[i,i]
  return L,U

In [23]:
Matriz_A = np.array([[16,4,8,4], [4,10,8,4], [8,8,12,10], [4,4,10,12]])
Matriz_b = np.array([32,26,38,30])
if verifca_LU(Matriz_A.copy(),Matriz_b.copy()):
  L,U = matrizes_L_U(Matriz_A.copy(),Matriz_b.copy())
  print(f"Matriz L:\n{L}")
  print(f"Matriz U:\n{U}")
else:
  print("A decomposição LU não é possível")

Matriz L:
[[1.         0.         0.         0.        ]
 [0.25       1.         0.         0.        ]
 [0.5        0.66666667 1.         0.        ]
 [0.25       0.33333333 1.5        1.        ]]
Matriz U:
[[16.  4.  8.  4.]
 [ 0.  9.  6.  3.]
 [ 0.  0.  4.  6.]
 [ 0.  0.  0.  1.]]


d) Se for possível resolver pelo método de Cholesky, apresente a matriz G obtida pelo método.

In [28]:
def matriz_G(A,b):
  lengh = len(b)
  #Criando a matriz G com mesmo tamanho de A
  G = np.zeros((lengh,lengh))
  for i in range (lengh):
    for j in range(i+1):
      soma=0.0
      if(i==j):
        for k in range(j):
          soma += G[i,k]**2
        G[i,j] = np.sqrt(A[i,i] - soma)
      else:
        for k in range(j):
          soma += G[i,k]*G[j,k]
        G[i,j]= (A[i,j] - soma)/G[j,j]
  return G

In [29]:
Matriz_A = np.array([[16,4,8,4], [4,10,8,4], [8,8,12,10], [4,4,10,12]])
Matriz_b = np.array([32,26,38,30])

if verifica_Cholesky(Matriz_A.copy(),Matriz_b.copy()):
  G = matriz_G(Matriz_A.copy(),Matriz_b.copy())
  print(f"Matriz G:\n{G}")
else:
  print("O uso do método de Cholesky não é possível para a matriz A.")

Matriz G:
[[4. 0. 0. 0.]
 [1. 3. 0. 0.]
 [2. 2. 2. 0.]
 [1. 1. 3. 1.]]
